# Parallel retrieval test for `StatelessCypherRetriever`

Three checks:

1. **Algorithmic equivalence** — `StatelessCypherRetriever` and the stateful `CypherNativeRetriever` produce identical results on the same questions. This validates that switching `_resource` from a graph property to a Cypher map variable preserves the algorithm.
2. **Concurrent correctness** — running the stateless retriever from multiple threads yields the same results as serial execution.
3. **Speedup** — wall-clock time decreases with parallel workers.

If the old `CypherNativeRetriever` were run concurrently, the shared `_resource` node property would cause queries to corrupt each other's state. This notebook demonstrates that `StatelessCypherRetriever` does not.

In [9]:
import os
import sys
import time
import pickle
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

from dotenv import load_dotenv
from neo4j import GraphDatabase

sys.path.insert(0, str(Path("..").resolve()))
load_dotenv("../.env")

True

In [10]:
from graph.embedder import Embedder
from graph.kin_rag import StatelessCypherRetriever, CypherNativeRetriever

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")

DATASET_NAME = "musique"
N_QUESTIONS = 30
MAX_WORKERS = 8

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
    max_connection_pool_size=MAX_WORKERS + 4,
)
embedder = Embedder(cache_path=Path("cache/embeddings_cache.pkl"))

QUERY_ENTITY_CACHE_PATH = Path(f"cache/query_entities-{DATASET_NAME}.pkl")
if QUERY_ENTITY_CACHE_PATH.exists():
    with open(QUERY_ENTITY_CACHE_PATH, "rb") as f:
        query_entity_cache = pickle.load(f)
    print(f"[QueryEntityCache] Loaded {len(query_entity_cache)} cached query entities")
else:
    query_entity_cache = {}
    print("[QueryEntityCache] No cache found, starting empty")

[Embedder] Loaded 828 cached embeddings
[QueryEntityCache] Loaded 999 cached query entities


In [11]:
with open(f"ground_truth-{DATASET_NAME}.pkl", "rb") as f:
    ground_truth = pickle.load(f)

questions = [gt["question"] for gt in ground_truth[:N_QUESTIONS]]
print(f"Testing on {len(questions)} questions from {DATASET_NAME}")

retriever_kwargs = dict(
    driver=driver,
    embedder=embedder,
    max_steps=3,
    decay=0.7,
    resource_threshold=0.01,
    seed_vector_fallback_k=5,
    top_k_entities=30,
    top_k_paths=30,
    query_entity_cache=query_entity_cache,
)

stateless_retriever = StatelessCypherRetriever(**retriever_kwargs)
stateful_retriever = CypherNativeRetriever(**retriever_kwargs)

Testing on 30 questions from musique


### Step 1: warm up caches

Run each question once serially to populate the query-entity LLM cache and the embedder cache, so the timing comparison below isolates Neo4j retrieval cost rather than LLM extraction cost.

In [12]:
missing = [q for q in questions if q not in query_entity_cache]
if missing:
    print(f"Warming up {len(missing)} questions (calling LLM)...")
    for q in missing:
        stateless_retriever._extract_query_entities(q)
    QUERY_ENTITY_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(QUERY_ENTITY_CACHE_PATH, "wb") as f:
        pickle.dump(query_entity_cache, f)
    print(f"Cached {len(missing)} new entries")
else:
    print("All questions already cached — LLM will not be called below")

All questions already cached — LLM will not be called below


### Step 2: algorithmic equivalence (Stateless vs CypherNative, both serial)

Each retriever must be run serially here — `CypherNativeRetriever` mutates the shared `_resource` property on graph nodes, so even if the test driver tried to interleave queries the Neo4j locks would serialize them anyway. We're not testing concurrency here, only that the two implementations agree.

If every fingerprint matches, the two are equivalent (modulo Cypher's tie-breaking, which is irrelevant because `fingerprint` uses sorted sets).

In [13]:
def fingerprint(result):
    """Order-independent signature of a RetrievalResult for equality testing."""
    return (
        tuple(sorted(result.seed_entities)),
        tuple(sorted(e["name"] for e in result.entities)),
        tuple(sorted(tuple(p["names"]) for p in result.paths)),
    )


t0 = time.perf_counter()
stateful_results = [stateful_retriever.retrieve(q) for q in questions]
stateful_elapsed = time.perf_counter() - t0
stateful_fingerprints = [fingerprint(r) for r in stateful_results]

t0 = time.perf_counter()
stateless_results = [stateless_retriever.retrieve(q) for q in questions]
stateless_elapsed = time.perf_counter() - t0
stateless_fingerprints = [fingerprint(r) for r in stateless_results]

matches = sum(s == p for s, p in zip(stateful_fingerprints, stateless_fingerprints))
mismatches = [
    i for i, (s, p) in enumerate(zip(stateful_fingerprints, stateless_fingerprints)) if s != p
]

print(f"CypherNative (stateful):   {stateful_elapsed:6.2f}s  ({stateful_elapsed / len(questions):.3f}s/q)")
print(f"Stateless:                 {stateless_elapsed:6.2f}s  ({stateless_elapsed / len(questions):.3f}s/q)")
print(f"\nFingerprint matches: {matches}/{len(questions)}")
if mismatches:
    print(f"Mismatch indices: {mismatches[:10]}{'...' if len(mismatches) > 10 else ''}")
    for idx in mismatches[:3]:
        print(f"\n--- Question {idx}: {questions[idx]} ---")
        sf = stateful_results[idx]
        sl = stateless_results[idx]
        print(f"  stateful seeds:   {sorted(sf.seed_entities)}")
        print(f"  stateless seeds:  {sorted(sl.seed_entities)}")
        sf_names = set(e["name"] for e in sf.entities)
        sl_names = set(e["name"] for e in sl.entities)
        print(f"  in stateful only:  {sorted(sf_names - sl_names)[:5]}")
        print(f"  in stateless only: {sorted(sl_names - sf_names)[:5]}")

CypherNative (stateful):    44.92s  (1.497s/q)
Stateless:                  11.04s  (0.368s/q)

Fingerprint matches: 29/30
Mismatch indices: [6]

--- Question 6: Who is the grandmother of Philippe, Duke of Orléans? ---
  stateful seeds:   ["françois d'orléans, prince of joinville", 'louis philippe', 'philippe', 'philippe i, duke of orléans', 'philippe ii, duke of orléans']
  stateless seeds:  ["françois d'orléans, prince of joinville", 'louis philippe', 'philippe', 'philippe i, duke of orléans', 'philippe ii, duke of orléans']
  in stateful only:  []
  in stateless only: []


### Step 3: concurrent execution of the stateless retriever

Compare against the serial stateless baseline measured above. Each worker count must produce fingerprints identical to that baseline.

In [14]:
def run_parallel(retriever, questions, max_workers):
    results = [None] * len(questions)
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        future_to_idx = {pool.submit(retriever.retrieve, q): i for i, q in enumerate(questions)}
        for fut in as_completed(future_to_idx):
            idx = future_to_idx[fut]
            results[idx] = fut.result()
    elapsed = time.perf_counter() - t0
    return results, elapsed


rows = []
for workers in (1, 2, 4, 8):
    parallel_results, parallel_elapsed = run_parallel(stateless_retriever, questions, workers)
    parallel_fingerprints = [fingerprint(r) for r in parallel_results]
    matches = sum(s == p for s, p in zip(stateless_fingerprints, parallel_fingerprints))
    speedup = stateless_elapsed / parallel_elapsed
    rows.append({
        "workers": workers,
        "elapsed_s": round(parallel_elapsed, 2),
        "speedup_vs_serial": round(speedup, 2),
        "matches_serial": f"{matches}/{len(questions)}",
    })
    print(f"  workers={workers:2d}  {parallel_elapsed:6.2f}s  speedup={speedup:.2f}x  matches={matches}/{len(questions)}")

  workers= 1   11.22s  speedup=0.98x  matches=30/30
  workers= 2    5.82s  speedup=1.90x  matches=30/30
  workers= 4    4.52s  speedup=2.44x  matches=30/30
  workers= 8    3.63s  speedup=3.04x  matches=30/30


### Summary

In [15]:
import pandas as pd

df = pd.DataFrame(rows)
df.insert(0, "mode", df["workers"].apply(lambda w: "serial" if w == 1 else f"parallel x{w}"))
print(df.to_string(index=False))

equivalence_ok = (matches == len(questions))
parallel_ok = all(row["matches_serial"].split("/")[0] == str(len(questions)) for row in rows)

print(f"\n[Equivalence vs CypherNative]  {matches}/{len(questions)} match  {'OK' if equivalence_ok else 'FAIL'}")
print(f"[Parallel correctness]         all worker counts produce identical results  {'OK' if parallel_ok else 'FAIL'}")

if equivalence_ok and parallel_ok:
    print("\nStatelessCypherRetriever is verified algorithmically equivalent to CypherNativeRetriever and parallel-safe.")
else:
    print("\nWARNING: one or more checks failed. Investigate.")

       mode  workers  elapsed_s  speedup_vs_serial matches_serial
     serial        1      11.22               0.98          30/30
parallel x2        2       5.82               1.90          30/30
parallel x4        4       4.52               2.44          30/30
parallel x8        8       3.63               3.04          30/30

[Equivalence vs CypherNative]  30/30 match  OK
[Parallel correctness]         all worker counts produce identical results  OK

StatelessCypherRetriever is verified algorithmically equivalent to CypherNativeRetriever and parallel-safe.


In [8]:
driver.close()